<a href="https://colab.research.google.com/github/thien2o7/AI/blob/main/GO_FARA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scikit-fuzzy numpy matplotlib

import gradio as gr
import folium
import pandas as pd
import re
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from datetime import datetime
from geopy.distance import geodesic
from geopy.geocoders import ArcGIS

# ==========================================
# 1. KHỞI TẠO BỘ NÃO FUZZY LOGIC (AI CORE)
# ==========================================
def build_fuzzy_system():
    # Biến đầu vào
    khoang_cach = ctrl.Antecedent(np.arange(0, 51, 1), 'khoang_cach') # 0-50 km
    thoi_tiet = ctrl.Antecedent(np.arange(0, 11, 1), 'thoi_tiet')     # 0-10 (0: Đẹp, 10: Bão)
    giao_thong = ctrl.Antecedent(np.arange(0, 101, 1), 'giao_thong')  # 0-100% kẹt xe
    khach_hang = ctrl.Antecedent(np.arange(1, 5.1, 0.1), 'khach_hang')# 1-5 sao đánh giá

    # Biến đầu ra
    he_so_gia = ctrl.Consequent(np.arange(0.8, 3.1, 0.1), 'he_so_gia') # Hệ số 0.8x -> 3.0x
    chiet_khau = ctrl.Consequent(np.arange(0, 31, 1), 'chiet_khau')    # Giảm 0% -> 30%

    # Hàm thành viên (Tập mờ)
    khoang_cach.automf(3, names=['ngan', 'vua', 'xa'])

    thoi_tiet['dep'] = fuzz.trimf(thoi_tiet.universe, [0, 0, 5])
    thoi_tiet['xau'] = fuzz.trimf(thoi_tiet.universe, [4, 10, 10])

    giao_thong['thong_thoang'] = fuzz.trimf(giao_thong.universe, [0, 0, 40])
    giao_thong['ket_xe'] = fuzz.trimf(giao_thong.universe, [30, 100, 100])

    khach_hang['thuong'] = fuzz.trimf(khach_hang.universe, [1, 1, 3.5])
    khach_hang['vip'] = fuzz.trimf(khach_hang.universe, [3.0, 5, 5])

    he_so_gia['re'] = fuzz.trimf(he_so_gia.universe, [0.8, 1.0, 1.2])
    he_so_gia['binh_thuong'] = fuzz.trimf(he_so_gia.universe, [1.0, 1.5, 2.0])
    he_so_gia['dat'] = fuzz.trimf(he_so_gia.universe, [1.8, 2.5, 3.0])

    chiet_khau['khong'] = fuzz.trimf(chiet_khau.universe, [0, 0, 5])
    chiet_khau['nhieu'] = fuzz.trimf(chiet_khau.universe, [10, 20, 30])

    # Các luật suy diễn
    rules = [
        ctrl.Rule(thoi_tiet['xau'] | giao_thong['ket_xe'], he_so_gia['dat']),
        ctrl.Rule(thoi_tiet['dep'] & giao_thong['thong_thoang'], he_so_gia['re']),
        ctrl.Rule(khoang_cach['xa'] & giao_thong['thong_thoang'], he_so_gia['binh_thuong']),
        ctrl.Rule(khach_hang['vip'], chiet_khau['nhieu']),
        ctrl.Rule(khach_hang['thuong'], chiet_khau['khong'])
    ]

    return ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

fuzzy_sim = build_fuzzy_system()

# ==========================================
# 2. KHỞI TẠO HỆ THỐNG GEOLOCATION & DATABASE
# ==========================================
geolocator = ArcGIS()
lich_su_chuyen_di = []

def get_coordinates(input_data):
    """Xử lý thông minh: Nhận cả Tọa độ số lẫn Địa chỉ chữ"""
    match = re.match(r"^\s*([+-]?\d+\.\d+)\s*,\s*([+-]?\d+\.\d+)\s*$", input_data)
    if match: return (float(match.group(1)), float(match.group(2)))
    try:
        location = geolocator.geocode(input_data + ", Việt Nam", timeout=10)
        if location: return (location.latitude, location.longitude)
    except: pass
    return None

# ==========================================
# 3. BỘ XỬ LÝ TRUNG TÂM (BACKEND)
# ==========================================
def he_thong_dat_xe_ai(diem_don_str, diem_den_str, loai_xe, muc_do_mua, muc_do_ket_xe, sao_khach_hang):
    global lich_su_chuyen_di

    # 1. Định vị tọa độ
    toado_don = get_coordinates(diem_don_str)
    toado_den = get_coordinates(diem_den_str)
    if not toado_don: return "⚠️ Lỗi: Không tìm thấy điểm đón.", None, gr.update()
    if not toado_den: return "⚠️ Lỗi: Không tìm thấy điểm đến.", None, gr.update()
    if toado_don == toado_den: return "⚠️ Lỗi: Điểm đón và điểm đến trùng nhau!", None, gr.update()

    khoang_cach_km = geodesic(toado_don, toado_den).km

    # 2. GỌI TRÍ TUỆ NHÂN TẠO (FUZZY LOGIC) TÍNH TOÁN
    # Truyền dữ liệu vào hệ thống mờ
    fuzzy_sim.input['khoang_cach'] = min(khoang_cach_km, 50)
    fuzzy_sim.input['thoi_tiet'] = muc_do_mua
    fuzzy_sim.input['giao_thong'] = muc_do_ket_xe
    fuzzy_sim.input['khach_hang'] = sao_khach_hang

    fuzzy_sim.compute()

    he_so_ai = fuzzy_sim.output['he_so_gia']
    chiet_khau_ai = fuzzy_sim.output['chiet_khau']

    # 3. Tính tiền
    gia_mo_cua = 12000
    gia_moi_km = 6000
    he_so_xe = {"🚲 Xe máy (Bike)": 1.0, "🚗 Ô tô 4 chỗ (Car)": 2.0, "🚙 Ô tô 7 chỗ (SUV)": 2.5}

    gia_goc = (gia_mo_cua + (khoang_cach_km * gia_moi_km)) * he_so_xe[loai_xe]
    tong_tien = gia_goc * he_so_ai * (1 - chiet_khau_ai/100)
    tong_tien = max(0, int(tong_tien))

    thoi_gian_phut = int((khoang_cach_km / 35) * 60 * (1 + muc_do_ket_xe/100)) # Kẹt xe thì đi lâu hơn

    # 4. Lưu Database
    lich_su_chuyen_di.append({
        "Thời gian": datetime.now().strftime("%d/%m %H:%M"),
        "Điểm đón": diem_don_str[:15] + "...",
        "Điểm đến": diem_den_str[:15] + "...",
        "Loại xe": loai_xe.split(" ")[1],
        "Khoảng cách": f"{khoang_cach_km:.1f} km",
        "Hệ số Surge": f"{he_so_ai:.1f}x",
        "Chiết khấu": f"-{int(chiet_khau_ai)}%",
        "Tổng tiền": f"{tong_tien:,} VNĐ"
    })
    df_lich_su = pd.DataFrame(lich_su_chuyen_di)

    # 5. Vẽ Bản đồ
    trung_tam = [(toado_don[0] + toado_den[0])/2, (toado_don[1] + toado_den[1])/2]
    ban_do = folium.Map(location=trung_tam, zoom_start=14)
    folium.Marker(toado_don, tooltip="📍 Điểm Đón", icon=folium.Icon(color="green", icon="play")).add_to(ban_do)
    folium.Marker(toado_den, tooltip="🏁 Điểm Đến", icon=folium.Icon(color="red", icon="stop")).add_to(ban_do)
    folium.PolyLine([toado_don, toado_den], color="blue", weight=4, opacity=0.7).add_to(ban_do)

    # 6. Xuất Biên lai
    ket_qua_md = f"""
### ✅ ĐÃ TÌM THẤY TÀI XẾ!
* **📍 Từ:** `{diem_don_str}`
* **🏁 Đến:** `{diem_den_str}`
* **📏 Quãng đường:** `{khoang_cach_km:.2f} km` | **⏳ Đi khoảng:** `{thoi_gian_phut} phút`
---
* 🧠 **Phân tích từ AI:** Hệ số giá tăng `x{he_so_ai:.2f}` (do điều kiện môi trường), Giảm giá thành viên `{chiet_khau_ai:.1f}%`
## 💵 TỔNG TIỀN: `{tong_tien:,} VNĐ`
    """
    return ket_qua_md, ban_do._repr_html_(), df_lich_su

# ==========================================
# 4. GIAO DIỆN FRONTEND (GRADIO)
# ==========================================
custom_css = """
.output_box { border: 3px solid #10b981 !important; border-radius: 15px !important; background-color: #ecfdf5 !important; padding: 15px; }
.gr-button-primary { background-color: #10b981 !important; border-radius: 10px !important; font-weight: bold !important; }
"""

app = gr.Blocks(theme=gr.themes.Default(primary_hue="emerald", secondary_hue="amber"), css=custom_css)

with app:
    gr.Markdown("# 🚖 GO-FARA: AI SURGE PRICING EDITION")

    with gr.Tabs():
        # TAB 1: MÀN HÌNH ĐẶT XE
        with gr.TabItem("🏠 Đặt Xe Ngay"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 🗺️ Lộ Trình (Chữ hoặc Tọa độ GPS)")
                    diem_don_input = gr.Textbox(label="Lên xe tại đâu?", value="Chợ Bến Thành, Quận 1")
                    diem_den_input = gr.Textbox(label="Về đâu?", value="Landmark 81")
                    loai_xe_input = gr.Radio(["🚲 Xe máy (Bike)", "🚗 Ô tô 4 chỗ (Car)", "🚙 Ô tô 7 chỗ (SUV)"], label="Phương tiện", value="🚗 Ô tô 4 chỗ (Car)")

                    gr.Markdown("### ☁️ Điều kiện môi trường (Fuzzy Inputs)")
                    muc_do_mua = gr.Slider(0, 10, value=2, step=1, label="Mức độ mưa/bão (0: Nắng đẹp -> 10: Bão)")
                    muc_do_ket_xe = gr.Slider(0, 100, value=30, step=1, label="Mức độ kẹt xe (%)")
                    sao_khach_hang = gr.Slider(1, 5, value=4.5, step=0.1, label="Hạng khách hàng (1-5 sao)")

                    dat_xe_btn = gr.Button("🚀 AI TÍNH TOÁN & GỌI XE", variant="primary", size="lg")

                with gr.Column(scale=1):
                    ket_qua_output = gr.Markdown(label="Biên lai", elem_classes="output_box")
                    ban_do_output = gr.HTML(label="Bản đồ")

        # TAB 2: LỊCH SỬ
        with gr.TabItem("🕒 Lịch Sử Chuyến Đi"):
            gr.Markdown("### 📚 Hệ thống lưu trữ dữ liệu AI")
            lich_su_output = gr.Dataframe(headers=["Thời gian", "Điểm đón", "Điểm đến", "Loại xe", "Khoảng cách", "Hệ số Surge", "Chiết khấu", "Tổng tiền"], interactive=False)

    dat_xe_btn.click(
        fn=he_thong_dat_xe_ai,
        inputs=[diem_don_input, diem_den_input, loai_xe_input, muc_do_mua, muc_do_ket_xe, sao_khach_hang],
        outputs=[ket_qua_output, ban_do_output, lich_su_output]
    )

app.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 11.2 MB/s eta 0:00:00


/tmp/ipykernel_4414/2047170876.py:154: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  app = gr.Blocks(theme=gr.themes.Default(primary_hue="emerald", secondary_hue="amber"), css=custom_css)
/tmp/ipykernel_4414/2047170876.py:154: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  app = gr.Blocks(theme=gr.themes.Default(primary_hue="emerald", secondary_hue="amber"), css=custom_css)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f901ca52e7d76953f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
